# 00 Inventory

**Owner:** Member B. **Goal:** confirm what the BBBD files actually contain before any pipeline relies on it: folder layout, file names, sampling rates, channels, quiz format. Results fill the Dataset Inventory and settle the UNVERIFIED items in `config.yaml` and `bbbd/io.py`.

Run top to bottom; commit with outputs.

## Setup (copy this cell to the top of every notebook)

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/bbbd_mml"):
        !git clone https://github.com/pbhatt0022/bbbd_mml.git /content/bbbd_mml
    else:
        !git -C /content/bbbd_mml pull
    %cd /content/bbbd_mml
    !pip install -q -e .

from bbbd.config import load_config, path
from bbbd import runlog, io
cfg = load_config()            # locally: load_config(data_root="D:/MML_BBBD")
print(cfg["paths"]["raw"])

## 1. Which experiments are on Drive?

In [ ]:
from pathlib import Path
raw = Path(cfg["paths"]["raw"])
print(sorted(p.name for p in raw.iterdir()) if raw.exists() else f"Missing: {raw}")

## 2. File inventory per experiment

One row per data file with its sidecar sampling rate and columns.

In [ ]:
import pandas as pd
run = runlog.start_run(cfg, name="inventory", notebook="00_inventory", member="B")
tables = {}
for exp in cfg["scope"]["eye_experiments"]:
    try:
        root = io.experiment_root(raw, exp)
    except FileNotFoundError as e:
        print(e); continue
    inv = io.inventory(root)
    inv.insert(0, "experiment", exp)
    tables[exp] = inv
    print(f"Exp {exp}: {inv.subject.nunique()} subjects, {len(inv)} files")
inv_all = pd.concat(tables.values(), ignore_index=True) if tables else pd.DataFrame()
inv_all.head(20)

## 3. Sampling rates (settles the 128 Hz question)

In [ ]:
if len(inv_all):
    display(inv_all.groupby(["experiment", inv_all.file.str.extract(r'_([a-z]+)\.(?:tsv\.gz|bdf)$')[0].rename("suffix")])["sfreq"].value_counts())

## 4. One EEG file in detail: channels and rate

In [ ]:
bdfs = io.find(io.experiment_root(raw, 2), "*.bdf") if tables else []
if bdfs:
    from bbbd import eeg
    r = eeg.load_bdf(bdfs[0])
    print(bdfs[0].name, "| sfreq:", r.info["sfreq"], "| n channels:", len(r.ch_names))
    print(r.ch_names)

## 5. Participants and quiz (phenotype) files

In [ ]:
for exp, inv in tables.items():
    root = io.experiment_root(raw, exp)
    print(f"--- Exp {exp}: participants.tsv rows = {len(io.read_participants(root))}")
    for f in io.phenotype_files(root):
        df = pd.read_csv(f, sep="\t")
        print(f"  {f.name}: {df.shape}, columns: {list(df.columns)[:12]}")

## 6. Save and register

In [ ]:
if len(inv_all):
    runlog.save_table(run, inv_all, "file_inventory.csv")
runlog.finish_run(run, note="File inventory; settles sampling rate + quiz format")

## 7. Record what you found

- Update the **Dataset Inventory** (participants per experiment, sampling rate per modality, quiz format).
- Update `config.yaml` (`signals.expected_sfreq_hz`, `eye.screen`) and `bbbd/io.py` naming if needed.
- Add a Decision Log row for anything that changes the plan; add an inference-log row if a planned analysis is affected.